In [6]:
import torch
from torch import nn
import polars as pl
import numpy as np
from sklearn.preprocessing import StandardScaler

In [7]:
# ==========================================
# 0 Hyperparameters
# ==========================================

MAX_RUL = 130
WINDOW_SEQ = 30
BATCH_SIZE = 32
HIDDEN_SIZE = 256
NUM_LAYERS = 2
NUM_OF_WORKERS = 0
LR = 1e-5
EPOCHS = 40
WEIGHT_DECAY = 5e-5
T0            = 30         # cosine annealing period
T_MULT        = 2          # cosine annealing multiplier


In [8]:
# ==========================================
# 1 LOAD DATA
# ==========================================

col_names = ["unit", "cycle"] + [f'op_{i}' for i in range(3)] + [f"s_{i}" for i in range(21)]
drop_cols = ["op_0","op_1","op_2", "s_0", "s_4", "s_15", "s_17", "s_18"]
feature_cols = [c for c in col_names if c not in ["unit", "cycle"] + drop_cols]

def load_all_data():

    train_lazy_df = pl.scan_csv(
        "CMAPSSData/train_FD00*.txt",
        separator=" ",
        truncate_ragged_lines=True,
        has_header=False,
        new_columns=col_names,
        include_file_paths="file_path"
    )

    train_df_load = train_lazy_df.select(col_names + ["file_path"]).with_columns(
        (pl.col("cycle").max().over(["file_path", "unit"]) - pl.col("cycle"))
        .clip(upper_bound=MAX_RUL)
        .alias("RUL")
    ).collect()


    test_labels_lazy = pl.scan_csv(
        "CMAPSSData/RUL_FD00*.txt",
        separator=" ",
        truncate_ragged_lines=True,
        has_header=False,
        include_file_paths="file_path"
    ).select([
        pl.col("file_path"),
        # Generujemy numer silnika (1, 2, 3...) dla każdego pliku z osobna
        pl.int_range(1, pl.len() + 1).over("file_path").alias("unit"),
        pl.col("column_1").alias("true_end_rul")
    ]).collect()


    test_lazy_df = pl.scan_csv(
        "CMAPSSData/test_FD00*.txt",
        separator=" ",
        truncate_ragged_lines=True,
        has_header=False,
        new_columns=col_names,
        include_file_paths="file_path"
    ).select(col_names + ["file_path"]).collect()

    return train_df_load, test_lazy_df, test_labels_lazy

In [9]:
# ==========================================
# 2 Creat windows
# ==========================================

def train_windows(df, feature_cols, window=WINDOW_SEQ):

    X, y = [], []

    for _, group_df in df.group_by(["file_path", "unit"]):
        group_df = group_df.sort("cycle")
        data = group_df[feature_cols].to_numpy()
        labels = group_df["RUL"].to_numpy()
        # print(CMAPSSData.shape, labels.shape)
        for i in range(len(group_df)-window+1):
            X.append(data[i:i+window])
            y.append(labels[i+window-1])
        # print(X.shape, y.shape)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def test_windows(df, labels, feature_cols, window=WINDOW_SEQ):

    X, y = [], []
    for i, (_, group_df) in enumerate(df.group_by(["file_path", "unit"])):
        group_df  = group_df.sort("cycle")
        data = group_df[feature_cols].to_numpy()
        if len(data) >= window:
            X.append(data[-window:])
        else:
            #pad shorter seq
            pad = np.zeros((window - len(data), len(feature_cols)))
            X.append(np.vstack([pad, data]))
        y.append(labels[i])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

In [10]:
# ==========================================
# 3 Create Datasets
# ==========================================

from torch.utils.data import Dataset, DataLoader

class Train_Dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, item):
        return self.X[item], self.y[item]

class Test_Dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, item):
        return self.X[item], self.y[item]


In [11]:
# ==========================================
# 4 Model
# ==========================================
class GRU(nn.Module):
    def __init__(self, num_layers, input_size, hidden_size):
        super().__init__()
        self.num_layers = num_layers
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, 1)

    # def forward(self, X):
    #     h0 = torch.zeros(self.num_layers, X.shape[0], self.hidden_size, device=X.device)
    #     out, hn = self.gru(X, h0)
    #     return self.fc(out[:, -1, :]).squeeze(-1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers = 1):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, 1)

    # def forward(self, X):
    #     h0 = torch.zeros(self.num_layers, X.shape[0], self.hidden_size, device=X.device)
    #     out, hn = self.gru(X, h0)
    #     return self.fc(out[:, -1, :]).squeeze(-1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

In [12]:
class gradient_explosion:
    def __init__(self, model):
        self.grad = []
        self.model = model

    def get_grad(self):
        total_norm = 0
        for p in self.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
        self.grad.append(total_norm ** 0.5)

    def print_grad_norm(self):
        print(f"mean Gradient norm: {np.mean(self.grad):.6f}, max: {np.max(self.grad):.6f}, min: {np.min(self.grad):.6f}")


In [13]:
# ==========================================
# 5 Trainer function
# ==========================================
def MSE(y_hat, y):
    return torch.mean((y_hat - y) ** 2)


class Trainer:
    def __init__(self, model, train_dataloader, test_dataloader, optimizer, scheduler, device, loss_fn, epoch, check_gradient = False):
        self.model = model
        self.train_dataloader = train_dataloader
        self.test_dataloader = test_dataloader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.loss_fn = loss_fn          # e.g., MSE for training
        self.epoch = epoch
        if check_gradient:
            self.gradient_checker = gradient_explosion(self.model)
        else:
            self.gradient_checker = None

    def fit_epoch(self):
        self.model.train()
        total_loss = 0.0
        total_samples = 0
        total_squared_error = 0.0

        for X, y in self.train_dataloader:
            X, y = X.to(self.device), y.to(self.device)

            self.optimizer.zero_grad(set_to_none=True)
            pred = self.model(X).squeeze()   # ensure shape (batch,)
            loss = self.loss_fn(pred, y)     # MSE
            loss.backward()
            self.optimizer.step()
            self.gradient_checker.get_grad()
            batch_size = y.size(0)
            total_loss += loss.item() * batch_size          # sum of MSE losses (weighted by batch size)
            total_squared_error += torch.sum((pred - y) ** 2).item()
            total_samples += batch_size

        # Return average MSE and RMSE for the epoch (optional)
        avg_mse = total_loss / total_samples
        rmse = np.sqrt(total_squared_error / total_samples)
        self.gradient_checker.print_grad_norm()
        return avg_mse, rmse

    def validate_epoch(self):
        self.model.eval()
        total_squared_error = 0.0
        total_samples = 0

        with torch.no_grad():
            for X, y in self.test_dataloader:
                X, y = X.to(self.device), y.to(self.device)
                pred = self.model(X).squeeze()
                total_squared_error += torch.sum((pred - y) ** 2).item()
                total_samples += y.size(0)

        rmse = np.sqrt(total_squared_error / total_samples)
        return rmse

    def fit(self):
        for epoch in range(self.epoch):
            train_mse, train_rmse = self.fit_epoch()
            val_rmse = self.validate_epoch()
            self.scheduler.step()

            print(f"Epoch {epoch+1}/{self.epoch} | Train MSE: {train_mse:.4f} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")

In [14]:
# Load
if __name__ == "__main__":

    train_df, test_df, RUL_test = load_all_data()

    train_df, train_labels = train_windows(train_df, feature_cols)
    test_df, test_labels = test_windows(test_df, RUL_test["true_end_rul"], feature_cols)

    scaler = StandardScaler().fit(train_df.reshape(-1, len(feature_cols)))
    train_df = scaler.transform(train_df.reshape(-1, len(feature_cols))).reshape(-1, WINDOW_SEQ, len(feature_cols))
    test_df  = scaler.transform(test_df.reshape(-1, len(feature_cols))).reshape(-1, WINDOW_SEQ, len(feature_cols))

    train_dataloader = DataLoader(Train_Dataset(train_df, train_labels), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_OF_WORKERS, pin_memory=True, drop_last=True)
    test_dataloader = DataLoader(Test_Dataset(test_df, test_labels), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_OF_WORKERS, pin_memory=True)

    loss_fn = MSE
    model = LSTM(len(feature_cols), HIDDEN_SIZE, NUM_LAYERS)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimiser = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    # Cosine Annealing with Warm Restarts
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimiser, T_0=T0, T_mult=T_MULT, eta_min=LR * 0.01
    )

    trainer = Trainer(
        model=model,
        train_dataloader=train_dataloader,
        test_dataloader=test_dataloader,
        optimizer=optimiser,
        scheduler=scheduler,
        device=device,
        loss_fn=loss_fn,
        epoch=EPOCHS,
        check_gradient=True
    )

    trainer.fit()

mean Gradient norm: 2134.853217, max: 3137.889292, min: 173.268573
Epoch 1/40 | Train MSE: 7308.4227 | Train RMSE: 85.4893 | Val RMSE: 80.0341
mean Gradient norm: 2054.142339, max: 3137.889292, min: 173.268573
Epoch 2/40 | Train MSE: 5711.4689 | Train RMSE: 75.5743 | Val RMSE: 72.1338
mean Gradient norm: 1914.809808, max: 3137.889292, min: 173.268573
Epoch 3/40 | Train MSE: 4519.8677 | Train RMSE: 67.2300 | Val RMSE: 65.1754
mean Gradient norm: 1762.723255, max: 3137.889292, min: 173.268573
Epoch 4/40 | Train MSE: 3571.6034 | Train RMSE: 59.7629 | Val RMSE: 59.4012
mean Gradient norm: 1607.874974, max: 3137.889292, min: 116.785548
Epoch 5/40 | Train MSE: 2858.9808 | Train RMSE: 53.4694 | Val RMSE: 55.0875
mean Gradient norm: 1454.824787, max: 3137.889292, min: 0.592926
Epoch 6/40 | Train MSE: 2366.5532 | Train RMSE: 48.6472 | Val RMSE: 52.3892
mean Gradient norm: 1308.555652, max: 3137.889292, min: 0.571121
Epoch 7/40 | Train MSE: 2072.1651 | Train RMSE: 45.5210 | Val RMSE: 51.2185
mea

In [10]:
train_df, test_df, RUL_test = load_all_data()
print("Train DataFrame shape:", train_df.shape)
print(train_df.head())
print("Feature columns:", feature_cols)
print("Number of feature columns expected:", len(feature_cols))

train_df, train_labels = train_windows(train_df, feature_cols)
test_df, test_labels = test_windows(test_df, RUL_test["true_end_rul"], feature_cols)
print(train_df.shape, test_df.shape)

scaler = StandardScaler().fit(train_df.reshape(-1, len(feature_cols)))
train_df = scaler.transform(train_df.reshape(-1, len(feature_cols))).reshape(-1, WINDOW_SEQ, len(feature_cols))
test_df  = scaler.transform(test_df.reshape(-1, len(feature_cols))).reshape(-1, WINDOW_SEQ, len(feature_cols))
print(test_df.shape)
print("Train feature mean after scaling:", train_df.reshape(-1, len(feature_cols)).mean(axis=0))
print("Train feature std after scaling:", train_df.reshape(-1, len(feature_cols)).std(axis=0))

Train DataFrame shape: (160359, 28)
shape: (5, 28)
┌──────┬───────┬─────────┬─────────┬───┬───────┬─────────┬────────────────────────────┬─────┐
│ unit ┆ cycle ┆ op_0    ┆ op_1    ┆ … ┆ s_19  ┆ s_20    ┆ file_path                  ┆ RUL │
│ ---  ┆ ---   ┆ ---     ┆ ---     ┆   ┆ ---   ┆ ---     ┆ ---                        ┆ --- │
│ i64  ┆ i64   ┆ f64     ┆ f64     ┆   ┆ f64   ┆ f64     ┆ str                        ┆ i64 │
╞══════╪═══════╪═════════╪═════════╪═══╪═══════╪═════════╪════════════════════════════╪═════╡
│ 1    ┆ 1     ┆ -0.0007 ┆ -0.0004 ┆ … ┆ 39.06 ┆ 23.419  ┆ CMAPSSData\train_FD001.txt ┆ 130 │
│ 1    ┆ 2     ┆ 0.0019  ┆ -0.0003 ┆ … ┆ 39.0  ┆ 23.4236 ┆ CMAPSSData\train_FD001.txt ┆ 130 │
│ 1    ┆ 3     ┆ -0.0043 ┆ 0.0003  ┆ … ┆ 38.95 ┆ 23.3442 ┆ CMAPSSData\train_FD001.txt ┆ 130 │
│ 1    ┆ 4     ┆ 0.0007  ┆ 0.0     ┆ … ┆ 38.88 ┆ 23.3739 ┆ CMAPSSData\train_FD001.txt ┆ 130 │
│ 1    ┆ 5     ┆ -0.0019 ┆ -0.0002 ┆ … ┆ 38.9  ┆ 23.4044 ┆ CMAPSSData\train_FD001.txt ┆ 130 │
└──────┴─

In [11]:
train_df.shape

(139798, 30, 16)

In [12]:
train_df, test_df, RUL_test = load_all_data()

train_df, train_labels = train_windows(train_df, feature_cols)
test_df, test_labels = test_windows(test_df, RUL_test["true_end_rul"], feature_cols)

scaler = StandardScaler().fit(train_df.reshape(-1, len(feature_cols)))
train_df = scaler.transform(train_df.reshape(-1, len(feature_cols))).reshape(-1, WINDOW_SEQ, len(feature_cols))
test_df  = scaler.transform(test_df.reshape(-1, len(feature_cols))).reshape(-1, WINDOW_SEQ, len(feature_cols))


train_dataloader = DataLoader(Train_Dataset(train_df, train_labels), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_OF_WORKERS, pin_memory=True, drop_last=True)

model = LSTM(input_size=len(feature_cols), hidden_size=256, num_layers=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-4)
criterion = nn.MSELoss()

# Take a small batch
X_batch, y_batch = next(iter(train_dataloader))

for epoch in range(460):
    optimizer.zero_grad()
    pred = model(X_batch)
    loss = criterion(pred, y_batch)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        # print_grad_norm(model)
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 8578.2656
Epoch 20, Loss: 7477.3984
Epoch 40, Loss: 6848.3486
Epoch 60, Loss: 6441.3574
Epoch 80, Loss: 6087.2476
Epoch 100, Loss: 5763.1431
Epoch 120, Loss: 5468.9258
Epoch 140, Loss: 5198.0435
Epoch 160, Loss: 4947.5684
Epoch 180, Loss: 4715.4604
Epoch 200, Loss: 4500.1924
Epoch 220, Loss: 4300.5337
Epoch 240, Loss: 4115.4370
Epoch 260, Loss: 3943.0500
Epoch 280, Loss: 3781.9614
Epoch 300, Loss: 3633.6626
Epoch 320, Loss: 3497.2827
Epoch 340, Loss: 3370.0154
Epoch 360, Loss: 3252.9851
Epoch 380, Loss: 3146.2590
Epoch 400, Loss: 3048.9604
Epoch 420, Loss: 2960.3276
Epoch 440, Loss: 2879.7019


In [13]:
train_df, test_df, RUL_test = load_all_data()
print("Train DataFrame shape:", train_df.shape)
print(train_df.head())
print("Feature columns:", feature_cols)
print("Number of feature columns expected:", len(feature_cols))

train_df, train_labels = train_windows(train_df, feature_cols)
test_df, test_labels = test_windows(test_df, RUL_test["true_end_rul"], feature_cols)

Train DataFrame shape: (160359, 28)
shape: (5, 28)
┌──────┬───────┬─────────┬─────────┬───┬───────┬─────────┬────────────────────────────┬─────┐
│ unit ┆ cycle ┆ op_0    ┆ op_1    ┆ … ┆ s_19  ┆ s_20    ┆ file_path                  ┆ RUL │
│ ---  ┆ ---   ┆ ---     ┆ ---     ┆   ┆ ---   ┆ ---     ┆ ---                        ┆ --- │
│ i64  ┆ i64   ┆ f64     ┆ f64     ┆   ┆ f64   ┆ f64     ┆ str                        ┆ i64 │
╞══════╪═══════╪═════════╪═════════╪═══╪═══════╪═════════╪════════════════════════════╪═════╡
│ 1    ┆ 1     ┆ -0.0007 ┆ -0.0004 ┆ … ┆ 39.06 ┆ 23.419  ┆ CMAPSSData\train_FD001.txt ┆ 130 │
│ 1    ┆ 2     ┆ 0.0019  ┆ -0.0003 ┆ … ┆ 39.0  ┆ 23.4236 ┆ CMAPSSData\train_FD001.txt ┆ 130 │
│ 1    ┆ 3     ┆ -0.0043 ┆ 0.0003  ┆ … ┆ 38.95 ┆ 23.3442 ┆ CMAPSSData\train_FD001.txt ┆ 130 │
│ 1    ┆ 4     ┆ 0.0007  ┆ 0.0     ┆ … ┆ 38.88 ┆ 23.3739 ┆ CMAPSSData\train_FD001.txt ┆ 130 │
│ 1    ┆ 5     ┆ -0.0019 ┆ -0.0002 ┆ … ┆ 38.9  ┆ 23.4044 ┆ CMAPSSData\train_FD001.txt ┆ 130 │
└──────┴─

In [14]:
train_df, test_df, RUL_test = load_all_data()
print("Train DataFrame shape:", train_df.shape)
train_df.head()

Train DataFrame shape: (160359, 28)


unit,cycle,op_0,op_1,op_2,s_0,s_1,s_2,s_3,s_4,s_5,s_6,s_7,s_8,s_9,s_10,s_11,s_12,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,file_path,RUL
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64,f64,str,i64
1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.7,1400.6,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.419,"""CMAPSSData\train_FD001.txt""",130
1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.0,23.4236,"""CMAPSSData\train_FD001.txt""",130
1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.2,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,"""CMAPSSData\train_FD001.txt""",130
1,4,0.0007,0.0,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,"""CMAPSSData\train_FD001.txt""",130
1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.0,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.8,8.4294,0.03,393,2388,100.0,38.9,23.4044,"""CMAPSSData\train_FD001.txt""",130


In [15]:
print(drop_cols)

['op_0', 'op_1', 'op_2', 's_0', 's_4', 's_15', 's_17', 's_18']
